[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C73_3D_Representation_Course/02_pointnet/02_pointnet.ipynb)

# 02 · 置换不变性与 max-pool

这个 notebook 把 PointNet 的四个**结构性事实**验证成恒等式或精确数值：

1. **置换不变**：max/min 的逐位差 **恰好 0.0**（选择操作）；
   而 **sum/mean 只是「数学上」不变**——浮点加法不满足结合律，相对差 1.43e-15。
2. **临界点集**：$N{=}1024$、$D{=}64$ 时只有 **24 个点**决定输出，
   删掉其余 1000 个点，输出**逐位相等**；
   而 $N$ 从 256 涨到 16384（64 倍），临界点数几乎不动（16 → 24）。
3. **三种聚合各自编码什么**：线性探针给出
   $\sum$ 的 $R^2{=}1.0000$、$\max$ 的 $0.7578$、mean 的 $0.1465$——
   **max 并没有「完全丢弃点数」**。
4. **分割头的上界**：坐标相同的两个点，无论邻域多不同，
   PointNet 的逐点特征**必然逐位相等**。

> 全程随机权重、不训练。**这些性质与权重取值无关**——
> 所以它们比任何精度数字都稳定。

## 0 · 环境与一个 30 行的 PointNet

In [ ]:
import numpy as np

print('numpy', np.__version__)
print('本 notebook 全程随机权重，**不训练** —— 验证的是结构性质')

N, D = 1024, 64
rng = np.random.default_rng(0)

def make_mlp(din, dout, seed=0, scale=0.3):
    r = np.random.default_rng(seed)
    return r.normal(size=(din, dout)), r.normal(size=dout) * scale

W, B = make_mlp(3, D, seed=1)

def pointwise(P, W=W, B=B):
    '''逐点 MLP（一层 + ReLU）-> (N, D)。'''
    return np.maximum(np.atleast_2d(P) @ W + B, 0.0)

def aggregate(F, kind='max'):
    return {'max': F.max(0), 'sum': F.sum(0), 'mean': F.mean(0)}[kind]

def pointnet(P, kind='max'):
    return aggregate(pointwise(P), kind)

P = rng.normal(size=(N, 3))
print(f'\n点数 {N}，特征维 {D}')
print(f'全局特征范数: max {np.linalg.norm(pointnet(P,"max")):.3f}  '
      f'sum {np.linalg.norm(pointnet(P,"sum")):.3f}  '
      f'mean {np.linalg.norm(pointnet(P,"mean")):.3f}')

## 1 · 置换不变性：一个恒等式

**注意差是恰好 0.0，不是「很小」**——因为 max/sum/mean 对置换是数学上不变的。

In [ ]:
print(f"{'聚合':>6s} {'20 次置换的最大逐位差':>22s} {'相对量级':>12s}")
inv = {}
for kind in ['max', 'min', 'sum', 'mean']:
    F0 = pointwise(P)
    ref = {'max': F0.max(0), 'min': F0.min(0),
           'sum': F0.sum(0), 'mean': F0.mean(0)}[kind]
    worst = 0.0
    for t in range(20):
        perm = np.random.default_rng(t).permutation(N)
        F = pointwise(P[perm])
        got = {'max': F.max(0), 'min': F.min(0),
               'sum': F.sum(0), 'mean': F.mean(0)}[kind]
        worst = max(worst, float(np.abs(got - ref).max()))
    scale = float(np.abs(ref).max())
    rel = worst / scale if scale > 0 else 0.0
    inv[kind] = (worst, rel)
    print(f'{kind:>6s} {worst:21.3e} {rel:11.2e}')

# ① max / min 是**选择**操作，逐位不变
assert inv['max'][0] == 0.0, 'max 必须逐位不变（它不做算术）'
assert inv['min'][0] == 0.0, 'min 同理'
# ② sum / mean 数学上不变，但浮点加法不满足结合律
EPS = np.finfo(np.float64).eps
assert 0 < inv['sum'][1] < 100 * EPS,     f"sum 的相对差应当是数值噪声量级，实测 {inv['sum'][1]:.2e}"
assert 0 < inv['mean'][1] < 100 * EPS
print(f'\n✅ **max / min 逐位不变；sum / mean 只是「数学上」不变**')
print(f'   sum 的相对差 {inv["sum"][1]:.2e} ≈ {inv["sum"][1]/EPS:.1f}ε '
      f'（ε = {EPS:.2e}）—— 纯数值噪声')
print('   原因：numpy 用 pairwise summation，累加的配对顺序依赖元素位置')
print('   → **置换不变性的单元测试不能一律用「恰好相等」**：')
print('     max 型可以（非零差 = 代码里有顺序依赖）；sum/mean 型必须用相对容差')

# 反例：不对称的聚合
def first_k(F, k=8):
    return F[:k].ravel()

g0 = first_k(pointwise(P))
g1 = first_k(pointwise(P[np.random.default_rng(3).permutation(N)]))
print(f'\n对照：取前 8 个点（不对称）-> 置换后差 = {np.abs(g1-g0).max():.3f}')
assert np.abs(g1 - g0).max() > 0.1, '不对称的聚合必然被置换破坏'
print('✅ 所以「对称」不是一个可选的设计偏好，它是这个问题的硬约束')

## 2 · 三种聚合在两种扰动下的表现

**两种扰动给出相反的排序**——这是本节的要点。

In [ ]:
def rel_change(a, b):
    return float(np.linalg.norm(a - b) / np.linalg.norm(b))

# 扰动一：随机丢点（点数变，密度分布不变）
print('扰动一：随机丢点\n')
print(f"{'保留比例':>9s} {'max':>10s} {'sum':>10s} {'mean':>10s}")
drop = {}
for frac in [0.75, 0.5, 0.25, 0.1]:
    k = int(N * frac)
    row = {}
    for kind in ['max', 'sum', 'mean']:
        g = pointnet(P, kind)
        ch = [rel_change(pointnet(P[np.random.default_rng(t).choice(N, k, replace=False)],
                                  kind), g) for t in range(20)]
        row[kind] = float(np.mean(ch))
    drop[frac] = row
    print(f'{frac:8.0%} {row["max"]:10.4f} {row["sum"]:10.4f} {row["mean"]:10.4f}')

# sum 的变化 ≈ 1 − 保留比例
for frac in [0.75, 0.5, 0.25]:
    assert abs(drop[frac]['sum'] - (1 - frac)) < 0.05, (frac, drop[frac]['sum'])
print('\n✅ sum 的变化 ≈ 1 − 保留比例（它与点数成正比）')
assert drop[0.5]['mean'] < drop[0.5]['max'] < drop[0.5]['sum']
print(f'   而 50% 丢点时：mean {drop[0.5]["mean"]:.4f} < '
      f'max {drop[0.5]["max"]:.4f} << sum {drop[0.5]["sum"]:.4f}')

# 扰动二：一半点聚成小簇（点数不变，密度分布大变）
print('\n扰动二：一半点聚成一个小簇（点数不变）\n')
P_clump = np.vstack([rng.normal(size=(N//2, 3)),
                     rng.normal(scale=0.05, size=(N//2, 3)) + np.array([2., 0., 0.])])
print(f"{'聚合':>6s} {'原范数':>10s} {'聚簇后范数':>12s} {'相对变化':>10s}")
clump = {}
for kind in ['max', 'mean']:
    g = pointnet(P, kind); g2 = pointnet(P_clump, kind)
    clump[kind] = rel_change(g2, g)
    print(f'{kind:>6s} {np.linalg.norm(g):10.3f} {np.linalg.norm(g2):11.3f} '
          f'{clump[kind]:10.4f}')

assert clump['max'] < clump['mean'], '密度扰动下 max 应当更稳'
print(f'\n✅ **两种扰动的排序相反**：')
print(f'   点数变化 -> mean 最稳（{drop[0.5]["mean"]:.4f}）')
print(f'   密度变化 -> max 最稳（{clump["max"]:.4f} vs mean {clump["mean"]:.4f}）')
print('   → 点云的主要扰动是**密度**（模块 01：520 倍衰减），所以选 max')

## 2b · 线性探针：哪种聚合编码了「点数」

比「输出变了多少」更直接的问法：**能不能从聚合向量里<em>读出</em> $N$？**

In [ ]:
def probe_count(kind, Ns=(128, 256, 384, 512, 768, 1024, 1536, 2048),
                trials=40, D_=D):
    '''线性回归探针：从聚合特征预测点数 N，返回 (R², MAE)。'''
    X, Y = [], []
    for t in range(trials):
        for n in Ns:
            Q = np.random.default_rng(1000 + t * 100 + n).normal(size=(n, 3))
            X.append(aggregate(pointwise(Q), kind))
            Y.append(float(n))
    A = np.column_stack([np.array(X), np.ones(len(X))])
    Y = np.array(Y)
    w, *_ = np.linalg.lstsq(A, Y, rcond=None)
    pred = A @ w
    r2 = 1 - ((Y - pred) ** 2).sum() / ((Y - Y.mean()) ** 2).sum()
    return float(r2), float(np.abs(Y - pred).mean())

print(f"{'聚合':>6s} {'R²':>9s} {'平均绝对误差(点)':>17s}")
pr = {}
for kind in ['max', 'sum', 'mean']:
    r2, mae = probe_count(kind)
    pr[kind] = r2
    print(f'{kind:>6s} {r2:9.4f} {mae:16.1f}')

assert pr['sum'] > 0.999, 'sum 应当精确编码点数'
assert pr['mean'] < 0.3, 'mean 应当几乎不编码点数'
assert 0.5 < pr['max'] < 0.95, f'max 应当**部分**编码点数，实测 {pr["max"]:.4f}'
print(f'\n✅ sum 精确编码（R²={pr["sum"]:.4f}）· '
      f'max **部分**编码（R²={pr["max"]:.4f}）· mean 几乎不编码（R²={pr["mean"]:.4f}）')
print('   max 的 0.76 不是噪声：**从更多样本里取最大值，期望更大**')
print('   —— 极值统计量本身依赖样本量，所以 max 通过分布偏移间接泄漏了点数')

# 拼接不是「兼得」，是「被范数大的那一半支配」
def cat_agg(Q):
    F = pointwise(Q)
    return np.concatenate([F.max(0), F.mean(0)])

g_cat = cat_agg(P)
ch_drop = np.mean([rel_change(cat_agg(P[np.random.default_rng(t).choice(N, N//2,
                              replace=False)]), g_cat) for t in range(20)])
ch_clump = rel_change(cat_agg(P_clump), g_cat)
print(f'\n拼接 [max, mean]: 丢一半点 {ch_drop:.4f}（max {drop[0.5]["max"]:.4f} / '
      f'mean {drop[0.5]["mean"]:.4f}）')
print(f'                  一半聚簇 {ch_clump:.4f}（max {clump["max"]:.4f} / '
      f'mean {clump["mean"]:.4f}）')
assert abs(ch_drop - drop[0.5]['max']) < abs(ch_drop - drop[0.5]['mean']), \
    '拼接的行为应当更靠近 max'
nm, nn = np.linalg.norm(pointnet(P,'max')), np.linalg.norm(pointnet(P,'mean'))
print(f'原因：max 的范数是 mean 的 {nm/nn:.1f} 倍，所以相对变化被它主导')
print('✅ 要真正兼得必须先各自归一化 —— 而那引入一个新超参数')

## 3 · 临界点集：本模块的核心

In [ ]:
def critical_set(P, W=W, B=B):
    '''返回决定 max-pool 输出的点的下标（去重）。'''
    F = pointwise(P, W, B)
    return np.unique(F.argmax(0))

crit = critical_set(P)
F_all = pointwise(P)
g = F_all.max(0)

# 两种「只用临界点」的算法，而它们的数值行为不同：
g_select = F_all[crit].max(0)                 # ① 从**同一个** F 里选
g_recomp = pointwise(P[crit]).max(0)          # ② **重算**子集的特征再取 max

print(f'N={N}, D={D}: 临界点 {len(crit)} 个（{100*len(crit)/N:.2f}%），'
      f'上界 min(D,N)={min(D,N)}')
print(f'  ① 从同一个 F 里选：最大逐位差 = {np.abs(g - g_select).max():.3e}')
print(f'  ② 重算子集特征  ：最大逐位差 = {np.abs(g - g_recomp).max():.3e}')
assert len(crit) <= min(D, N), '临界点数不可能超过 min(D, N)'
assert np.abs(g - g_select).max() == 0.0, '①「选择」是恒等式，必须逐位相等'
EPS = np.finfo(np.float64).eps
assert np.abs(g - g_recomp).max() <= 8 * EPS, '②「重算」只保证到几个 ULP'
print(f'\n✅ 删掉其余 {N - len(crit)} 个点（{100*(N-len(crit))/N:.1f}%），输出不变')
print('   但要区分两件事：')
print('   ① **从同一个特征矩阵里选** —— 逐位相等，这是恒等式')
print('   ② **重算子集的特征** —— 只保证到几个 ULP')
print('      因为 BLAS 对不同形状的矩阵乘用不同的分块/向量化路径')
print('      （本课在 N=128, D=16 上量到 2ε 的差）')
print('   → **「逐位相等」的断言必须说清「重算了什么」**')

# 临界点数不随 N 增长
print(f"\n{'N':>7s} {'D':>5s} {'临界点数':>9s} {'占 N 的比例':>12s}")
sizes = {}
for n in [256, 1024, 4096, 16384]:
    for d in [64, 256]:
        Wd, Bd = make_mlp(3, d, seed=6)
        Q = np.random.default_rng(5).normal(size=(n, 3))
        c = len(np.unique(pointwise(Q, Wd, Bd).argmax(0)))
        sizes[(n, d)] = c
        print(f'{n:7d} {d:5d} {c:9d} {100*c/n:11.2f}%')

# N 涨 64 倍，临界点数几乎不动
assert sizes[(16384, 64)] < 3 * sizes[(256, 64)], \
    f'N 涨 64 倍，临界点数不该按比例涨（{sizes[(256,64)]} -> {sizes[(16384,64)]}）'
assert 100 * sizes[(16384, 64)] / 16384 < 1.0
print(f'\n✅ N 从 256 到 16384（64 倍），临界点数 '
      f'{sizes[(256,64)]} → {sizes[(16384,64)]}（几乎不动）')
print(f'   占比从 {100*sizes[(256,64)]/256:.2f}% 掉到 '
      f'{100*sizes[(16384,64)]/16384:.2f}%')
print('   → **全局 max-pool 只「看到」几十个极值点，与输入点数无关**')
print('   → 这同时解释了「对点丢失鲁棒」与「抓不住细节」两件事')

## 4 · 单点删除实验：$\max$ 与 $\sum$ 的信息结构

In [ ]:
for kind in ['max', 'sum']:
    g0 = pointnet(P, kind)
    unchanged, min_ch = 0, np.inf
    for i in range(N):
        keep = np.ones(N, bool); keep[i] = False
        d = float(np.abs(pointnet(P[keep], kind) - g0).max())
        if d == 0.0:
            unchanged += 1
        min_ch = min(min_ch, d)
    print(f'{kind:>5s}: 删掉单点后输出**完全不变**的点数 = {unchanged}/{N}'
          f'   最小变化量 = {min_ch:.3e}')
    if kind == 'max':
        assert unchanged == N - len(crit), \
            f'不变的点数应当恰好等于 N − |临界集| = {N - len(crit)}'
    else:
        assert unchanged == 0 and min_ch > 0.1

print(f'\n✅ max: {N-len(crit)}/{N} 个点可删除（= N − |临界集|，精确吻合）')
print('   sum: 0/1024 —— 每一个点都影响输出')
print('   → max 把点集压成「若干极值点」，sum 压成「所有点的总量」')
print('   → 而点云里「点数」主要由距离决定（模块 01），所以它是**噪声**')
print('     于是「丢弃点数信息」从缺点变成了优点')

## 5 · 置换不变 ≠ 旋转不变

In [ ]:
def rot_z(deg):
    t = np.deg2rad(deg); c, s = np.cos(t), np.sin(t)
    return np.array([[c, -s, 0.], [s, c, 0.], [0., 0., 1.]])

g0 = pointnet(P, 'max')
crit0 = set(critical_set(P).tolist())
print(f"{'绕 z 旋转':>10s} {'全局特征相对变化':>18s} {'临界点集变了几个点':>20s}")
for deg in [0, 5, 15, 45, 90, 180]:
    Q = P @ rot_z(deg).T
    ch = rel_change(pointnet(Q, 'max'), g0)
    c = set(critical_set(Q).tolist())
    print(f'{deg:9d}° {ch:17.4f} {len(crit0 ^ c):19d}')
    if deg == 0:
        assert ch == 0.0 and len(crit0 ^ c) == 0

ch15 = rel_change(pointnet(P @ rot_z(15).T, 'max'), g0)
assert ch15 > 0.03, f'15° 旋转应当明显改变输出，实测 {ch15:.4f}'
print(f'\n✅ 15° 旋转就让全局特征变 {ch15:.1%} —— **PointNet 不是旋转不变的**')
print('   而在自动驾驶里通常不需要 T-Net：外参已经把点云摆正了（C72）')
print('   → 不变性来自结构（免费、可靠）比来自数据（要学、可能失效）更好')

## 5b · 分割头的数学上界

全局特征对所有点是**同一个**向量，所以它无法区分任何两个点。
**于是逐点判别力全部来自 $h(x_i)$，而 $h$ 看不到邻域。**

In [ ]:
def seg_features(P, W=W, B=B):
    '''PointNet 分割头的输入：[逐点特征 || 全局特征]。'''
    F = pointwise(P, W, B)
    g = F.max(0)
    return np.concatenate([F, np.broadcast_to(g, (len(F), len(g)))], axis=1)

# 构造两个坐标完全相同、但邻域截然不同的点
TARGET = np.array([0.5, -0.2, 1.1])
scene_a = np.vstack([TARGET, rng.normal(size=(200, 3))])                    # 稀疏邻域
scene_b = np.vstack([TARGET,
                     TARGET + rng.normal(scale=0.02, size=(200, 3))])       # 密集邻域

fa, fb = seg_features(scene_a)[0], seg_features(scene_b)[0]
print(f'同一个坐标 {TARGET}，两种截然不同的邻域：')
print(f'  逐点特征部分（前 {D} 维）最大逐位差 = {np.abs(fa[:D]-fb[:D]).max():.3e}')
print(f'  全局特征部分（后 {D} 维）最大逐位差 = {np.abs(fa[D:]-fb[D:]).max():.3e}')
assert np.abs(fa[:D] - fb[:D]).max() == 0.0, \
    '逐点特征只依赖坐标，必须**逐位相等**'
assert np.abs(fa[D:] - fb[D:]).max() > 0.0, '而全局特征会变（场景不同）'
print('\n✅ **逐点特征逐位相等** —— 邻域信息完全没有进入')
print('   所以 PointNet 的分割是「逐点分类 + 一个全局偏置」：')
print('   它能学「这个高度的点通常是标志」，学不到「周围有平面所以是地面」')

# 全局特征对所有点相同 -> 它不能区分点
S = seg_features(P)
assert np.allclose(S[:, D:], S[0, D:]), '全局特征那一半对所有点必须完全相同'
print(f'\n拼接后的维度: 逐点 {D} + 全局 {D} = {S.shape[1]}')
for d_local, d_global in [(64, 1024), (128, 1024)]:
    print(f'  常见配置 h={d_local}, g={d_global}: '
          f'**{100*d_global/(d_local+d_global):.0f}% 的维度对所有点相同**')
print('   → 这解释了为什么 PointNet 的分割对场景整体变化很敏感')

## 6 · 小结

| 结论 | 数值 |
|---|---|
| 置换不变 | **max/min 逐位差恰好 0.0**；sum/mean 只到 1.43e-15（≈6ε） |
| $\sum$ 对点数的敏感性 | 相对变化 ≈ $1-$ 保留比例（50% 丢点 → 0.498） |
| 两种扰动的排序**相反** | 点数变：mean 最稳；密度变：**max 最稳** |
| 线性探针 | $\sum$ 的 $R^2$ **1.0000** · $\max$ **0.7578** · mean **0.1465** |
| 拼接 $[\max,\text{mean}]$ | 不是兼得，而是被范数大的一半支配（46.9 vs 6.3） |
| **临界点集** | $N{=}1024,D{=}64$ → **24 个点**；删掉其余 1000 个**逐位不变** |
| 临界点数与 $N$ 无关 | $N$ 涨 64 倍，它从 16 → 24；占比 6.25% → **0.15%** |
| 单点删除 | max **1000/1024** 无影响；sum **0/1024** |
| 旋转 | 15° 就让全局特征变 **7.4%** |
| 分割头 | 坐标相同 → 逐点特征**逐位相等**（邻域信息进不来） |

## ✏️ 练习 1：对称性检查器

实现 `is_symmetric(agg_fn, n=64, d=8, trials=20, seed=0)`：
给一个作用在 `(N,D)` 特征矩阵上、返回一维向量的函数，
判断它是否置换不变。返回 `(bool, 最大逐位差)`。

要求用**恰好相等**判定。

> 注意：这会让 `sum` / `mean` / `cumsum-last` **判为不对称**——
> 而那是正确的行为。它们在<em>数学上</em>对称，但在浮点下不是逐位不变的
> （第 1 节量到相对差 1.43e-15）。
> **所以这个检查器测的是「逐位置换不变」，它对 max 型模型才是合适的判据。**

In [ ]:
def is_symmetric(agg_fn, n=64, d=8, trials=20, seed=0):
    """返回 (是否置换不变, 最大逐位差)。"""
    # TODO
    raise NotImplementedError

In [ ]:
# —— 自测 ——
CANDIDATES = {
    'max':        lambda F: F.max(0),
    'sum':        lambda F: F.sum(0),
    'mean':       lambda F: F.mean(0),
    'min':        lambda F: F.min(0),
    'max+mean':   lambda F: np.concatenate([F.max(0), F.mean(0)]),
    'sorted-top3':lambda F: np.sort(F, axis=0)[-3:].ravel(),
    'first-8':    lambda F: F[:8].ravel(),          # ❌ 不对称
    'cumsum-last':lambda F: np.cumsum(F, axis=0)[-1],
    'diff-first-last': lambda F: F[0] - F[-1],      # ❌ 不对称
}
print(f"{'候选':>18s} {'置换不变':>9s} {'最大逐位差':>12s}")
res = {}
for name, fn in CANDIDATES.items():
    ok, dev = is_symmetric(fn)
    res[name] = ok
    print(f'{name:>18s} {str(ok):>9s} {dev:12.3e}')

# 选择型（无算术）-> 逐位不变
assert res['max'] and res['min'], 'max/min 是选择操作，必须逐位不变'
assert res['sorted-top3'], '排序后取 top-k 也是选择操作（Deep Sets 里常用的一族）'
# 算术型 -> 数学上对称但**不逐位**不变
assert not res['sum'], 'sum 在浮点下不逐位不变（第 1 节：相对差 1.43e-15）'
assert not res['mean'], 'mean 同理'
assert not res['cumsum-last'], 'cumsum 的最后一项数学上等于 sum，浮点下同样不逐位'
# 真正不对称的
assert not res['first-8'] and not res['diff-first-last']
# 拼接：max+mean 里含 mean，所以整体也不逐位不变
assert not res['max+mean'], '拼接里只要含算术型聚合，整体就不逐位不变'
print('\n✅ 练习 1 通过。而这个检查器把候选分成了三类，而不是两类：')
print('   ① **选择型**（max / min / sorted-top-k）—— 逐位置换不变')
print('   ② **算术型**（sum / mean / cumsum）—— 数学上对称，浮点下不逐位')
print('   ③ **不对称**（first-8 / diff-first-last）—— 差是 O(1)，与浮点无关')
print('   区分 ② 与 ③ 要看差的**量级**：1e-15 是数值噪声，1e-1 是逻辑错误')

## 📖 参考答案 1

In [ ]:
# 练习 1 参考答案
def is_symmetric(agg_fn, n=64, d=8, trials=20, seed=0):
    r = np.random.default_rng(seed)
    F = r.normal(size=(n, d))
    ref = np.asarray(agg_fn(F), float)
    worst = 0.0
    for t in range(trials):
        perm = np.random.default_rng(seed + 1 + t).permutation(n)
        got = np.asarray(agg_fn(F[perm]), float)
        if got.shape != ref.shape:
            return False, float('inf')
        worst = max(worst, float(np.abs(got - ref).max()))
    return bool(worst == 0.0), worst

assert is_symmetric(lambda F: F.max(0))[0]
assert not is_symmetric(lambda F: F[:8].ravel())[0]
print('✅ 参考答案 1 通过')
print('   用「恰好 0」而不是「< 1e-9」：真正的对称函数在浮点下也是逐位相同的，')
print('   因为它对同一批数做同一批运算 —— 只有顺序变了，而这三种运算与顺序无关。')
print('   （注意 sum 也成立：numpy 的 add.reduce 对同一批数给同一结果。）')

## ✏️ 练习 2：临界点集分析器

实现 `critical_analysis(P, W, B)`，返回 dict：

- `'idx'` —— 临界点下标（升序去重）
- `'size'`, `'frac'` —— 大小与占比
- `'upper_bound'` —— $\min(D, N)$
- `'exact_select'` —— bool：从**同一个**特征矩阵里选临界点，输出是否逐位相等
  （**这是恒等式，必须为真**）
- `'exact_recompute'` —— bool：**重算**子集的特征再取 max，是否逐位相等
  （<em>不保证</em>——BLAS 对不同形状用不同路径）
- `'recompute_ulp'` —— 重算路径的最大差，以 $\varepsilon$ 为单位
- `'deletable'` —— 删掉任意单点后输出不变的点数
- `'consistent'` —— bool：`deletable == N - size`（**这是一个可验证的恒等式**）

In [ ]:
def critical_analysis(P, W, B):
    """返回 dict(idx, size, frac, upper_bound, exact_select,
    exact_recompute, recompute_ulp, deletable, consistent)。"""
    # TODO
    raise NotImplementedError

In [ ]:
# —— 自测 ——
for n, d in [(128, 16), (512, 64), (1024, 64)]:
    Wd, Bd = make_mlp(3, d, seed=11)
    Q = np.random.default_rng(12).normal(size=(n, 3))
    a = critical_analysis(Q, Wd, Bd)
    assert set(a) == {'idx', 'size', 'frac', 'upper_bound', 'exact_select',
                      'exact_recompute', 'recompute_ulp', 'deletable',
                      'consistent'}
    assert a['size'] <= a['upper_bound'] == min(d, n)
    assert a['exact_select'] is True, '「选择」是恒等式，必须逐位相等'
    assert a['recompute_ulp'] < 8.0, \
        f"重算路径的差应当只有几个 ULP，实测 {a['recompute_ulp']:.1f}ε"
    assert a['consistent'] is True, \
        f"deletable({a['deletable']}) 必须等于 N−size({n - a['size']})"
    print(f'N={n:5d} D={d:4d}: 临界点 {a["size"]:3d}（{a["frac"]:6.2%}）'
          f' 选择逐位 {str(a["exact_select"]):5s} 重算逐位 '
          f'{str(a["exact_recompute"]):5s}（{a["recompute_ulp"]:.1f}ε）'
          f' 可删除 {a["deletable"]:5d}')

# 占比随 N 下降
fr = []
for n in [128, 512, 2048, 8192]:
    Wd, Bd = make_mlp(3, 64, seed=11)
    Q = np.random.default_rng(12).normal(size=(n, 3))
    fr.append(critical_analysis(Q, Wd, Bd)['frac'])
print(f'\n占比随 N: ' + ' → '.join(f'{f:.2%}' for f in fr))
assert fr == sorted(fr, reverse=True), '占比必须随 N 单调下降'
assert fr[-1] < fr[0] / 10, '涨 64 倍点数，占比应当降一个数量级以上'
print('✅ 练习 2 通过。两个结论：')
print('   ① **deletable == N − |临界集|** 是一个恒等式（一个点可删除 ⟺ 它不是任何一维的 argmax）')
print('   ② 「选择」逐位相等，而「重算」只到几个 ULP —— '
      'BLAS 对不同形状用不同路径')

## 📖 参考答案 2

In [ ]:
# 练习 2 参考答案
def critical_analysis(P, W, B):
    P = np.asarray(P, float)
    F = pointwise(P, W, B)
    g = F.max(0)
    idx = np.unique(F.argmax(0))
    # ① 选择：从同一个 F 里取，这是恒等式
    sel_dev = float(np.abs(F[idx].max(0) - g).max())
    # ② 重算：BLAS 对不同形状可能走不同路径
    rec_dev = float(np.abs(pointwise(P[idx], W, B).max(0) - g).max())
    # 可删除性也用「选择」口径判定，才是恒等式
    deletable = 0
    for i in range(len(P)):
        keep = np.ones(len(P), bool); keep[i] = False
        if np.abs(F[keep].max(0) - g).max() == 0.0:
            deletable += 1
    eps = np.finfo(np.float64).eps
    return {'idx': idx, 'size': int(len(idx)), 'frac': float(len(idx)/len(P)),
            'upper_bound': int(min(F.shape[1], len(P))),
            'exact_select': bool(sel_dev == 0.0),
            'exact_recompute': bool(rec_dev == 0.0),
            'recompute_ulp': float(rec_dev / eps),
            'deletable': int(deletable),
            'consistent': bool(deletable == len(P) - len(idx))}

a = critical_analysis(P, W, B)
assert a['exact_select'] and a['consistent']
print(f'✅ 参考答案 2 通过（临界点 {a["size"]}，可删除 {a["deletable"]}，'
      f'N−size = {len(P)-a["size"]}）')
print('   两个实现细节：')
print('   ① `deletable` 必须用「从同一个 F 里选」来判 —— 用「重算」会被 ULP 噪声污染；')
print('   ② 所以恒等式 deletable == N−size 只在「选择」口径下严格成立。')

## ✏️ 练习 3：聚合函数的信息探针

实现 `agg_probe(agg_name, quantity, Ns=..., trials=20)`，
用线性探针量「聚合特征里能读出多少某个量」。
`quantity` 取 `'count'`（点数）或 `'scale'`（点云的整体尺度，用坐标标准差）。

返回 `(R², MAE)`。然后用它验证：

- `'count'`：$\sum$ 的 $R^2 \approx 1$、mean 的 $R^2$ 很低
- `'scale'`：**三种聚合都能读出尺度**（因为 $h$ 是逐点的、尺度直接进特征）

In [ ]:
def agg_probe(agg_name, quantity, Ns=(256, 512, 1024), scales=(0.5, 1.0, 2.0),
              trials=20):
    """返回 (R², MAE)。"""
    # TODO
    raise NotImplementedError

In [ ]:
# —— 自测 ——
print(f"{'聚合':>6s} {'读点数 R²':>11s} {'读尺度 R²':>11s}")
tab = {}
for kind in ['max', 'sum', 'mean']:
    r2c, _ = agg_probe(kind, 'count')
    r2s, _ = agg_probe(kind, 'scale')
    tab[kind] = (r2c, r2s)
    print(f'{kind:>6s} {r2c:11.4f} {r2s:11.4f}')

# ① 点数：sum 高、mean 低
assert tab['sum'][0] > 0.9, tab['sum'][0]
assert tab['mean'][0] < tab['sum'][0], '(mean 应当比 sum 更难读出点数)'

# ② 尺度：三种都能读出（因为逐点 MLP 直接吃坐标）
for kind in ['max', 'sum', 'mean']:
    assert tab[kind][1] > 0.5, f'{kind} 应当能读出尺度，实测 {tab[kind][1]:.3f}'
print('\n✅ 练习 3 通过：')
print('   **点数**是聚合方式的函数（sum 保留、mean 丢弃）；')
print('   **尺度**是逐点 MLP 就能编码的量，所以三种聚合都读得出。')
print('   → 所以「换聚合函数」影响的是<集合级>的量，不是<点级>的量')

## 📖 参考答案 3

In [ ]:
# 练习 3 参考答案
def agg_probe(agg_name, quantity, Ns=(256, 512, 1024), scales=(0.5, 1.0, 2.0),
              trials=20):
    X, Y = [], []
    for t in range(trials):
        for n in Ns:
            for sc in scales:
                Q = np.random.default_rng(7000 + t*97 + n + int(sc*10)).normal(
                    scale=sc, size=(n, 3))
                X.append(aggregate(pointwise(Q), agg_name))
                Y.append(float(n) if quantity == 'count'
                         else float(Q.std()))
    A = np.column_stack([np.array(X), np.ones(len(X))])
    Y = np.array(Y)
    w, *_ = np.linalg.lstsq(A, Y, rcond=None)
    pred = A @ w
    r2 = 1 - ((Y - pred)**2).sum() / ((Y - Y.mean())**2).sum()
    return float(r2), float(np.abs(Y - pred).mean())

assert agg_probe('sum', 'count')[0] > 0.9
assert agg_probe('mean', 'scale')[0] > 0.5
print('✅ 参考答案 3 通过')
print('   注意训练集里必须同时变 N 与 scale —— 否则两个量相关，探针分不开它们。')
print('   （这与 C10 的「混淆因子」是同一个问题：探针也需要正确的实验设计。）')

## ✏️ 练习 4：分割头的上界检查

实现 `seg_head_audit(target, scene_a, scene_b, W, B)`：
给同一个坐标 `target` 与两个不同的场景，返回 dict：

- `'local_identical'` —— bool：逐点特征是否**逐位相等**
- `'global_differs'` —— bool：全局特征是否不同
- `'global_shared'` —— bool：同一场景内全局特征对所有点是否相同
- `'global_frac'` —— 拼接后「对所有点相同」的维度占比

这三个 bool 全为真，就证明了「PointNet 的分割头无法使用局部结构」。

In [ ]:
def seg_head_audit(target, scene_a, scene_b, W, B):
    """返回 dict(local_identical, global_differs, global_shared, global_frac)。"""
    # TODO
    raise NotImplementedError

In [ ]:
# —— 自测 ——
TGT = np.array([0.5, -0.2, 1.1])
r4 = np.random.default_rng(21)
SA = np.vstack([TGT, r4.normal(size=(200, 3))])                       # 稀疏邻域
SB = np.vstack([TGT, TGT + r4.normal(scale=0.02, size=(200, 3))])     # 密集邻域
SC = np.vstack([TGT, TGT + r4.normal(scale=0.5, size=(200, 3))])      # 中等邻域

a = seg_head_audit(TGT, SA, SB, W, B)
assert set(a) == {'local_identical', 'global_differs', 'global_shared',
                  'global_frac'}
print('稀疏邻域 vs 密集邻域（同一个坐标）:')
for k, v in a.items():
    print(f'  {k:18s} = {v}')

assert a['local_identical'] is True, '逐点特征只依赖坐标 —— 必须逐位相等'
assert a['global_differs'] is True, '而全局特征会变（场景不同）'
assert a['global_shared'] is True, '同一场景内，全局特征对所有点相同'
assert abs(a['global_frac'] - 0.5) < 1e-12, f'D+D 拼接时应为 50%'

# 换第三个场景，结论不变
b = seg_head_audit(TGT, SA, SC, W, B)
assert b['local_identical'] is True and b['global_differs'] is True

print('\n三个 bool 全为真 -> **PointNet 的分割头在数学上无法使用局部结构**')
print('   它只能做「逐点分类 + 一个全局偏置」')
print('\n常见配置下「对所有点相同」的维度占比：')
for dl, dg in [(64, 1024), (128, 1024), (64, 64)]:
    print(f'   h={dl:4d}, g={dg:4d} -> {dg/(dl+dg):6.1%}')
print('✅ 练习 4 通过：这就是 PointNet++ 的分层聚合要补的那个能力')

## 📖 参考答案 4

In [ ]:
# 练习 4 参考答案
def seg_head_audit(target, scene_a, scene_b, W, B):
    def feats(scene):
        F = pointwise(scene, W, B)
        g = F.max(0)
        return F, g
    Fa, ga = feats(np.asarray(scene_a, float))
    Fb, gb = feats(np.asarray(scene_b, float))
    # target 是每个场景的第 0 个点
    local_same = bool(np.abs(Fa[0] - Fb[0]).max() == 0.0)
    global_diff = bool(np.abs(ga - gb).max() > 0.0)
    # 同一场景内，全局那一半对所有点相同
    S = np.concatenate([Fa, np.broadcast_to(ga, (len(Fa), len(ga)))], axis=1)
    d_local = Fa.shape[1]
    shared = bool(np.allclose(S[:, d_local:], S[0, d_local:]))
    return {'local_identical': local_same, 'global_differs': global_diff,
            'global_shared': shared,
            'global_frac': float(len(ga) / (d_local + len(ga)))}

a = seg_head_audit(TGT, SA, SB, W, B)
assert a['local_identical'] and a['global_differs'] and a['global_shared']
print('✅ 参考答案 4 通过')
print('   这三条一起构成一个**不可能性证明**：')
print('   全局那一半对所有点相同（不能区分点）+ 逐点那一半只看坐标（不含邻域）')
print('   ⇒ 任何依赖局部结构的判别都做不到。而这与权重取值无关。')

## 🧪 真实工程胶囊

```python
# ── 1) PyTorch 里的 PointNet 骨架（本课的 numpy 版本对应这几行）──
class PointNetEncoder(nn.Module):
    def __init__(self, d=1024):
        super().__init__()
        self.mlp = nn.Sequential(                 # ← 逐点：用 Conv1d(kernel=1) 实现
            nn.Conv1d(3, 64, 1), nn.BatchNorm1d(64), nn.ReLU(),
            nn.Conv1d(64, 128, 1), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Conv1d(128, d, 1), nn.BatchNorm1d(d), nn.ReLU())
    def forward(self, x):                          # x: (B, 3, N)
        f = self.mlp(x)                            # (B, d, N)
        return f.max(dim=2).values, f              # ← 全局特征 + 逐点特征
#   ⚠️ Conv1d(kernel_size=1) 就是「逐点 MLP」——这是保证置换不变的关键写法。
#      任何 kernel_size > 1 都会引入顺序依赖，而它不会报错（只会静默地破坏不变性）。

# ── 2) 临界点集：一行就能算，而且它是免费的诊断 ──
with torch.no_grad():
    _, f = enc(x)                                  # (B, d, N)
    crit = f.argmax(dim=2).unique(dim=-1)          # 每个通道的 argmax
print(f'临界点 {crit.numel()} / {x.shape[-1]}')
#   ↑ 用它回答「我把输入点数翻倍到底有没有用」——如果临界点数没变，就没用

# ── 3) 自动驾驶里不要加 T-Net ──
#   点云已经被外参摆正（C72），所以学一个 3×3 对齐只会引入分布外风险。
#   MMDetection3D 的 PointNet++ backbone 默认就没有 T-Net。

# ── 4) 分组用球查询而不是 kNN（第 6 节）──
from mmcv.ops import ball_query, knn
idx = ball_query(0.0, 0.8, 32, xyz, new_xyz)       # min_r, max_r, K
#   ⚠️ 稀疏区凑不满 K 个时会重复第一个点（padding），
#      所以组内特征里会有重复 —— 而 max-pool 恰好对重复免疫（第 4 节）。
#      这是一个「架构选择恰好兼容工程妥协」的例子，而它不是巧合：
#      球查询与 max-pool 是一起被设计出来的。
```

> **落地顺序建议**：先算一次临界点集（一行，立刻告诉你「加点数有没有用」），
> 再检查你的逐点 MLP 是否真的是 `kernel_size=1`（这是静默破坏不变性的头号写法），
> 最后才考虑换聚合函数——而换之前先用练习 3 的探针确认你想保留/丢弃的是哪个量。